# 0 Imports

In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from src.suporte import funcoes_suporte as fs

## 0.1 Funções Suporte

In [3]:
fs.jupyter_settings(altura = 10, largura = 12, fonte = 8)
fs.supressao_notacao(casa_decimal = 2)

# 0.2 Load Data

In [4]:
df_fe = fs.load_pickle("../data/interim/1.descricao.pkl")
df_fe.sample(5)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
174918,551858,21156,RETROSPOT CHILDRENS APRON,6,2011-05-04,1.95,18041,United Kingdom
296765,562932,71459,HANGING JAM JAR T-LIGHT HOLDER,12,2011-08-10,0.85,16904,United Kingdom
398166,571224,82483,WOOD 2 DRAWER CABINET WHITE FINISH,6,2011-10-14,6.95,13136,United Kingdom
217771,C555935,22432,WATERING CAN PINK BUNNY,-6,2011-06-08,1.95,12567,France
158218,550279,22520,CHILDS GARDEN TROWEL BLUE,1,2011-04-15,0.85,14700,United Kingdom


# 1.0 F.E.

In [5]:
df_ref = df_fe['customer_id'].drop_duplicates( ignore_index=True)
df_ref.head()

0    17850
1    13047
2    12583
3    13748
4    15100
Name: customer_id, dtype: int64

## 1.1 Gross Revenue (Faturamento) quantidade * preço

In [6]:
df_fe["faturamento"] = df_fe["quantity"] * df_fe["unit_price"]

## 1.2 Monetário

In [7]:
df_monetario = df_fe[["customer_id", "faturamento"]].groupby("customer_id").sum().reset_index()

df_ref = pd.merge(df_ref, df_monetario, how="left", on="customer_id")

df_ref.sample()

,customer_id,faturamento
1872,16495,684.41


In [8]:
df_ref.isna().sum()

customer_id    0
faturamento    0
dtype: int64

In [9]:
del df_monetario

## 1.3 Recência

In [10]:
df_recencia = df_fe[["customer_id", "invoice_date"]].groupby("customer_id").max().reset_index()

df_recencia["recencia_days"] = (df_fe["invoice_date"].max() - df_recencia["invoice_date"]).dt.days

df_recencia = df_recencia[["customer_id", "recencia_days"]].copy()

df_ref = pd.merge(df_ref, df_recencia, how = "left", on="customer_id")

df_ref.head()

,customer_id,faturamento,recencia_days
0,17850,5288.63,302
1,13047,3079.10,31
2,12583,7187.34,2
3,13748,948.25,95
4,15100,635.10,330


In [11]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
dtype: int64

In [12]:
del df_recencia

## 1.4 Frequência

In [13]:
df_frequencia = df_fe[["customer_id", "invoice_no"]].drop_duplicates().groupby("customer_id").count().reset_index().rename(columns = {"invoice_no": "frequencia"})

df_ref = pd.merge(df_ref, df_frequencia, how='left', on='customer_id')

df_ref.head()

,customer_id,faturamento,recencia_days,frequencia
0,17850,5288.63,302,35
1,13047,3079.10,31,18
2,12583,7187.34,2,18
3,13748,948.25,95,5
4,15100,635.10,330,6


In [14]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
frequencia       0
dtype: int64

In [15]:
del df_frequencia

## 1.4 Avg Ticket

In [16]:
avg_ticket = df_ref[['customer_id','faturamento']].groupby('customer_id').mean().reset_index().rename(columns = {'faturamento':'avg_faturamento'})
df_ref = pd.merge(df_ref, avg_ticket, how='left', on='customer_id')
df_ref.head()

,customer_id,faturamento,recencia_days,frequencia,avg_faturamento
0,17850,5288.63,302,35,5288.63
1,13047,3079.10,31,18,3079.10
2,12583,7187.34,2,18,7187.34
3,13748,948.25,95,5,948.25
4,15100,635.10,330,6,635.10


In [17]:
df_ref.isna().sum()

customer_id        0
faturamento        0
recencia_days      0
frequencia         0
avg_faturamento    0
dtype: int64

# 2.0 Exportar DF

In [19]:
path = "../data/interim/2.fe.pkl"
fs.save_pickle(obj=df_ref,path=path)